# Build a hierarchical spatial query taxonomy from embeddings

In [1]:
import pandas as pd
import numpy as np
import umap
import csv
from pathlib import Path
import hdbscan

In [2]:
# Load raw queries
queries = pd.read_csv("interim/queries.csv.zip")

# Load query embeddings
xb = np.load("interim/query_emb.npy").astype('float32')  # (n_queries, 384)

# Load classifier results
classifier_results = pd.read_csv('output/queries-classified.csv.zip')

In [3]:
queries.shape

(1010916, 2)

In [4]:
spatial_idx = classifier_results.is_spatial_pred.eq(1)

## UMAP + HDBSCAN

### Run grid search to find best parameters

1. We reduce embedding dimention with Umap before clustering (384 is too many for hsbscan)
2. No right way of doing it, but let's try 5-10-15 dimensions
3. Try a bunch of other params as well (grid search-like)

In [5]:
xb_spatial = xb[spatial_idx]
xb_spatial.shape

(104288, 384)

In [6]:
def log(path='output/umap_hdbscan_runs.csv', config_id=None):
    
    log_path = Path(path)
    write_header = not log_path.exists()
    
    with log_path.open('a', newline='') as f:
        writer = csv.writer(f)
    
        if write_header:
            writer.writerow([
                'config_id',
                'seed',
                'umap_dim',
                'umap_neighbours',
                'umap_min_dist',
                'min_cluster_size',
                'min_samples',
                'dbcv',
                'noise_frac',
                'n_clusters',
                'median_cluster_size',
                'cluster_selection_method'
            ])
    
        writer.writerow([
            config_id,
            seed,
            umap_dim,
            umap_neighbours,
            umap_min_dist,
            min_cluster_size,
            min_samples,
            dbcv,
            noise,
            n_clusters,
            median_size,
            cluster_selection_method
        ])

### Run grid search to find top-5 parameters

In [11]:
%%time

cluster_selection_method = 'eom' # by default, let's search the grid with eom

for umap_dim in (5, 10, 15):
    for umap_neighbours in (13, 30, 50):
        for umap_min_dist in (0.0, 0.1):

            for min_cluster_size in (25, 50, 100, 200):
                for min_samples in (None, 5, 10):

                    print('----')
                    print('UMAP n dims:', umap_dim)
                    print('UMAP n neighbours:', umap_neighbours)
                    print('UMAP min dist:', umap_min_dist)
                    print('HDBSCAN min cluster size:', min_cluster_size)
                    print('HDBSCAN min samples:', min_samples)
            
                    reducer = umap.UMAP(
                        n_components=umap_dim,
                        n_neighbors=umap_neighbours,
                        min_dist=umap_min_dist,
                        metric='cosine',
                        n_jobs=1, # -1 for all cores for faster processing but then cannot use random seed...
                        random_state=42
                    )
                    
                    xb_spatial_low = reducer.fit_transform(xb_spatial)
            
                    clusterer = hdbscan.HDBSCAN(
                        min_cluster_size=min_cluster_size,
                        min_samples=min_samples,
                        metric='euclidean',
                        cluster_selection_method=cluster_selection_method,
                        gen_min_span_tree=True
                    )

                    # Fit the model
                    labels = clusterer.fit_predict(xb_spatial_low)

                    # Calculate quality metrics
                    dbcv = clusterer.relative_validity_
                    noise = (labels == -1).mean() # -1 aka invalid/noise
                    
                    n_clusters = len(set(labels)) - (1 if -1 in labels else 0) # exclude noise -1 cluster
                    
                    vc = pd.Series(labels[labels != -1]).value_counts()
                    median_size = vc.median() if len(vc) else 0

                    print(f'DBCV Score: {dbcv}')
                    print(f'% invalid/noise/-1 clusters: {noise}')
                    print(f'# clusters: {n_clusters}')
                    print(f'median cluster size: {median_size}')
                    print('*****************')

                    # Log to the file
                    log()

----
UMAP n dims: 5
UMAP n neighbours: 13
UMAP min dist: 0.0
HDBSCAN min cluster size: 25
HDBSCAN min samples: None
DBCV Score: 0.2416853653441588
% invalid/noise/-1 clusters: 0.45013807916538817
# clusters: 480
median cluster size: 60.5
*****************
----
UMAP n dims: 5
UMAP n neighbours: 13
UMAP min dist: 0.0
HDBSCAN min cluster size: 25
HDBSCAN min samples: 5
DBCV Score: 0.19119169224845756
% invalid/noise/-1 clusters: 0.39415848419760663
# clusters: 699
median cluster size: 49.0
*****************
----
UMAP n dims: 5
UMAP n neighbours: 13
UMAP min dist: 0.0
HDBSCAN min cluster size: 25
HDBSCAN min samples: 10
DBCV Score: 0.19758450290896498
% invalid/noise/-1 clusters: 0.42727830622890456
# clusters: 622
median cluster size: 51.0
*****************
----
UMAP n dims: 5
UMAP n neighbours: 13
UMAP min dist: 0.0
HDBSCAN min cluster size: 50
HDBSCAN min samples: None
DBCV Score: 0.33264898840420576
% invalid/noise/-1 clusters: 0.42451672292114145
# clusters: 195
median cluster size: 1

## Check how stable clustering is

- Umap is stochastic
- Let's rerun umap+hdbscan for top config 5 times each to see how `dbcv` and other qa metrics differ
- We then have more confidence in picking the best params
- top-10 config gets dbcv >= 0.355, which is a nice initial threshold (and all noise < 0.42; between 43 and 174 clusters)

In [22]:
top_configs = pd.read_csv('output/umap_hdbscan_runs.csv').sort_values('dbcv', ascending=False).head(10)
top_configs

,seed,umap_dim,umap_neighbours,umap_min_dist,min_cluster_size,min_samples,dbcv,noise_frac,n_clusters,median_cluster_size,cluster_selection_method
202,42,15,50,0.0,200,5.0,0.422420,0.333979,71,576.0,eom
57,42,5,50,0.0,200,NaN,0.418498,0.359715,48,554.0,eom
201,42,15,50,0.0,200,NaN,0.390241,0.351335,48,570.0,eom
106,42,10,30,0.0,200,5.0,0.383788,0.328906,78,426.5,eom
131,42,10,50,0.0,200,10.0,0.378306,0.334161,75,517.0,eom
203,42,15,50,0.0,200,10.0,0.377357,0.336999,71,525.0,eom
178,42,15,30,0.0,200,5.0,0.373953,0.337153,75,450.0,eom
130,42,10,50,0.0,200,5.0,0.365142,0.343194,77,543.0,eom
195,42,15,50,0.0,50,NaN,0.355942,0.419780,174,147.5,eom
33,42,5,30,0.0,200,NaN,0.355120,0.381655,43,565.0,eom


In [24]:
%%time

for i, config in top_configs.iterrows():

    for cluster_selection_method in ['eom']:# 'leaf': # leaf produces slightly more clusters, slightly smaller clusters, but higher noise and lower dbcv, so let's ignore.
        
        for seed in [42, 0, 1, 2, 3, 4]: # check 42 again to make sure we can reproduce

            # get params into vars so logger knows what to write
            umap_dim = config['umap_dim']
            umap_neighbours = config['umap_neighbours']
            umap_min_dist = config['umap_min_dist']
            min_cluster_size = config['min_cluster_size']
            min_samples = None if pd.isna(config['min_samples']) else int(config['min_samples']) # np.nan and None (expected by hdbscan's min_samples) are not equivalent here so we need this extra check
    
            reducer = umap.UMAP(
                n_components=umap_dim,
                n_neighbors=umap_neighbours,
                min_dist=umap_min_dist,
                metric='cosine',
                n_jobs=1,
                random_state=seed
            )
    
            xb_spatial_low = reducer.fit_transform(xb_spatial)
                        
            clusterer = hdbscan.HDBSCAN(
                min_cluster_size=min_cluster_size,
                min_samples=min_samples, 
                metric='euclidean',
                cluster_selection_method=cluster_selection_method,
                gen_min_span_tree=True
            )

            # Fit the model
            labels = clusterer.fit_predict(xb_spatial_low)

            # Calculate quality metrics
            dbcv = clusterer.relative_validity_
            noise = (labels == -1).mean() # label=-1 for invalid/noise
            
            n_clusters = len(set(labels)) - (1 if -1 in labels else 0) # exclude noise (label=-1) cluster
            
            vc = pd.Series(labels[labels != -1]).value_counts()
            median_size = vc.median() if len(vc) else 0

            log('output/top_runs_different_seeds.csv', i)

CPU times: user 1h 46min 51s, sys: 59 s, total: 1h 47min 50s
Wall time: 1h 49min 4s


# Get cluster labels from the best performing set of umap+hdbscan params

## Let's summarise run consistency per each top config & which config to use

- Probably highest dbcv mean assuming it also has low noise fraction

In [6]:
top_runs = pd.read_csv('output/top_runs_different_seeds.csv')

top_runs_stats = top_runs.groupby('config_id').aggregate({
    'dbcv': ('max', 'mean', 'std'),
    'noise_frac': ('max', 'mean', 'std'),
    'n_clusters': ('max', 'mean', 'std')
}).round(3).sort_values([('dbcv', 'mean')], ascending=False)

top_runs_stats

dbcv               noise_frac               n_clusters           \
             max   mean    std        max   mean    std        max     mean   
config_id                                                                     
106        0.396  0.341  0.052      0.359  0.333  0.019         83   76.500   
131        0.382  0.338  0.049      0.378  0.348  0.019         80   75.000   
178        0.388  0.332  0.064      0.347  0.331  0.014         81   75.167   
201        0.390  0.332  0.051      0.405  0.370  0.020         51   47.833   
203        0.377  0.330  0.045      0.379  0.349  0.017         83   75.000   
202        0.422  0.324  0.055      0.359  0.347  0.010         79   75.500   
130        0.365  0.312  0.041      0.392  0.347  0.031         86   78.333   
195        0.356  0.311  0.029      0.436  0.427  0.007        180  177.167   
57         0.418  0.303  0.082      0.435  0.388  0.033         54   50.000   
33         0.355  0.292  0.044      0.439  0.403  0.023         48   45.000   

                  
             std  
config_id         
106        4.593  
131        3.033  
178        3.312  
201        1.941  
203        5.404  
202        3.082  
130        5.203  
195        3.488  
57         3.162  
33         1.789

In [10]:
# Ok let's re-label top run again to get the labels
top_runs.query(f'config_id == {top_runs_stats.iloc[0].name}')

,config_id,seed,umap_dim,umap_neighbours,umap_min_dist,min_cluster_size,min_samples,dbcv,noise_frac,n_clusters,median_cluster_size,cluster_selection_method
18,106,42,10,30,0.0,200,5.0,0.383788,0.328906,78,426.5,eom
19,106,0,10,30,0.0,200,5.0,0.295722,0.305193,71,468.0,eom
20,106,1,10,30,0.0,200,5.0,0.380119,0.327027,73,455.0,eom
21,106,2,10,30,0.0,200,5.0,0.314430,0.359150,83,466.0,eom
22,106,3,10,30,0.0,200,5.0,0.275548,0.348036,80,436.0,eom
23,106,4,10,30,0.0,200,5.0,0.395582,0.331102,74,489.0,eom


In [12]:
# Let's pick the seed with the highest dbcv
top_run = top_runs.query(f'config_id == {top_runs_stats.iloc[0].name}').sort_values('dbcv', ascending=False).iloc[0]
top_run

config_id                        106
seed                               4
umap_dim                          10
umap_neighbours                   30
umap_min_dist                    0.0
min_cluster_size                 200
min_samples                      5.0
dbcv                        0.395582
noise_frac                  0.331102
n_clusters                        74
median_cluster_size            489.0
cluster_selection_method         eom
Name: 23, dtype: object

In [15]:
%%time

reducer = umap.UMAP(
    n_components=int(top_run.umap_dim),
    n_neighbors=int(top_run.umap_neighbours),
    min_dist=top_run.umap_min_dist,
    metric='cosine',
    n_jobs=1,
    random_state=int(top_run.seed)
)
    
xb_spatial_low = reducer.fit_transform(xb_spatial)
            
clusterer = hdbscan.HDBSCAN(
    min_cluster_size=int(top_run.min_cluster_size),
    min_samples=int(top_run.min_samples), 
    metric='euclidean',
    cluster_selection_method='eom',
    gen_min_span_tree=True
)

# Get the lables
labels = clusterer.fit_predict(xb_spatial_low)

CPU times: user 1min 21s, sys: 690 ms, total: 1min 22s
Wall time: 1min 24s


In [17]:
print( len(labels) )
assert len(set(labels)) == (top_run.n_clusters + 1) # adding -1 for noise cluster

104288


## Manually review clusters

In [43]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.spatial.distance import cdist

In [57]:
def analyze_clusters(queries, labels, embeddings, top_n_words=20, sample_size=10):
    
    df = pd.DataFrame({'query': queries, 'label': labels})
    unique_labels = sorted(df['label'].unique())
    
    # 1. Class-based TF-IDF
    # Join all queries per cluster into a single 'document'
    docs_per_class = df.groupby('label')['query'].apply(lambda x: ' '.join(x)).reset_index()
    tfidf_model = TfidfVectorizer(stop_words='english', ngram_range=(1, 2))
    tfidf_matrix = tfidf_model.fit_transform(docs_per_class['query'])
    words = tfidf_model.get_feature_names_out()
    
    report = []

    for i, label in enumerate(unique_labels):
        # Get cluster indices
        idx = df[df['label'] == label].index
        cluster_embeddings = embeddings[idx]
        cluster_queries = df.loc[idx, 'query'].values
        
        # 2. Find Centroid (Representative Query)
        # Calculate mean embedding and find the query closest to it
        centroid_vector = cluster_embeddings.mean(axis=0).reshape(1, -1)
        distances = cdist(centroid_vector, cluster_embeddings, 'cosine')[0]
        centroid_idx = distances.argmin()
        representative_query = cluster_queries[centroid_idx]
        
        # 3. Get Top TF-IDF words
        row_data = tfidf_matrix.getrow(i).toarray()[0]
        top_word_indices = row_data.argsort()[-top_n_words:][::-1]
        top_terms = [words[ti] for ti in top_word_indices]
        
        # 4. Sample Queries
        sample = df.loc[idx, 'query'].sample(min(len(idx), sample_size), random_state=42).tolist()
        
        # Build Markdown Section
        section = [
            f'## Cluster {label} ({len(idx)} samples)',
            f'**Representative Query:** `{representative_query}`',
            f'**Top Terms:** {", ".join(top_terms)}',
            '\n**Sample Queries:**',
            '\n'.join([f'- {q}' for q in sample]),
            '\n---'
        ]
        report.append('\n'.join(section))
        
    return '\n'.join(report)

In [58]:
markdown_output = analyze_clusters(
    queries[spatial_idx].reset_index().query_text,
    labels,
    xb[spatial_idx]
)

In [59]:
with open('output/cluster_review.md', 'w') as f:
    f.write(markdown_output)